# 🔭 Module 4 — Production Tracing & Experimentation (Arize Phoenix)

Modules 1–3 scored hand-built test cases and hand-rolled `trace` lists. In production nobody writes the trace for you: the agent emits it, a tracing backend stores it, and evaluation runs **over the stored spans**. This notebook does that end to end with [Arize Phoenix](https://docs.arize.com/phoenix) — the fourth framework in this course, next to DeepEval, RAGAS and MLflow — on a small customer-support agent built here, so nothing depends on code outside this notebook.

## Learning Objectives
In this notebook, you will learn:
1. **Tracing an agent** - OpenTelemetry/OpenInference spans for the LLM call, the router decision and each tool, viewed in the Phoenix UI
2. **Router and skill evaluation** - an LLM judge on the router's tool choice and a code check on a tool's output, both logged back onto the spans
3. **Trajectory evaluation** - the convergence score (optimal path length ÷ actual path length) computed from the stored spans
4. **Experiments** - a versioned dataset, a task and evaluators, run twice to compare two system prompts

## Prerequisites
- The Phoenix extra in the root environment: `uv pip install --python .venv/bin/python -e '.[phoenix]'` (Phoenix 20.x, `arize-phoenix-evals` 3.x, `arize-phoenix-client` 3.x)
- A running Phoenix server: `phoenix serve` in a terminal, then open http://localhost:6006. Set `PHOENIX_COLLECTOR_ENDPOINT` only if it runs elsewhere
- `OPENAI_API_KEY` in `.env`. **Cost: key** — every agent run and every judge call is a paid `gpt-4o-mini` call (a few cents for the whole notebook)
- Modules 2–3 ([`09_Tool_Use_Evaluation.ipynb`](09_Tool_Use_Evaluation.ipynb), [`12_Agent_Trajectory_Evaluation.ipynb`](12_Agent_Trajectory_Evaluation.ipynb)) for the metric ideas this notebook moves onto stored traces

---
## ⚙️ Part 0: Setup

`register()` points an OpenTelemetry tracer provider at Phoenix, and `auto_instrument=True` loads every installed OpenInference instrumentor — here, the OpenAI one, so each chat completion becomes an `LLM` span with its messages, tool calls and token counts. The `Client` is the read side: it queries spans back out and writes evaluations onto them.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: tracing, Phoenix client, OpenAI
# ============================================================================
import json
import os
import warnings
from collections import Counter
from datetime import datetime

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
from phoenix.client import Client
from phoenix.otel import register

warnings.filterwarnings("ignore")
load_dotenv(find_dotenv())
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env before running this notebook."

PROJECT = "c01-13-support-agent"   # one Phoenix project per notebook keeps the UI readable
MODEL = "gpt-4o-mini"

tracer_provider = register(project_name=PROJECT, auto_instrument=True)
tracer = tracer_provider.get_tracer(__name__)   # OpenInference tracer: @tracer.agent / .chain / .tool
phoenix = Client()                               # defaults to http://localhost:6006
openai_client = OpenAI()

print(f"✅ Tracing to Phoenix project '{PROJECT}'")

---
## 🧭 Part 1: Trace an agent

### 1.1 🤖 A small support agent

Three tools over mock data, and a loop: the model either calls a tool or answers. Each piece gets its own span kind, which is what makes the trace evaluable later:

- `AGENT` — one per request (`run_agent`), the root of the trace
- `CHAIN` named `router` — one per decision, with the question as input and the chosen tool as output
- `LLM` — created automatically by the OpenAI instrumentor inside the router
- `TOOL` — one per tool call (`@tracer.tool`)

In [ ]:
# ============================================================================
# MOCK DATA AND TOOLS
# ============================================================================
ORDERS = {
    "1042": {"status": "shipped", "carrier": "UPS", "eta": "2 days", "delivered": False},
    "2077": {"status": "delivered", "carrier": "DHL", "eta": None, "delivered": True},
}
POLICY = {
    "returns": "Items can be returned within 30 days of delivery, unused, with the receipt.",
    "refunds": "Refunds go back to the original payment method within 5-7 business days.",
    "shipping": "We ship to the US and Canada. Canada orders take 5-9 business days.",
}


@tracer.tool
def lookup_order(order_id: str) -> str:
    """Look up an order's status by its ID."""
    order = ORDERS.get(order_id)
    return json.dumps(order) if order else json.dumps({"error": f"order {order_id} not found"})


@tracer.tool
def search_policy(topic: str) -> str:
    """Return the policy text for a topic: returns, refunds or shipping."""
    return POLICY.get(topic.lower(), "No policy found for that topic.")


@tracer.tool
def escalate(reason: str) -> str:
    """Hand the conversation to a human agent."""
    return json.dumps({"ticket": "ESC-1", "reason": reason})


TOOLS = {"lookup_order": lookup_order, "search_policy": search_policy, "escalate": escalate}
TOOL_SCHEMAS = [
    {"type": "function", "function": {"name": "lookup_order", "description": "Look up an order's status by its ID.",
     "parameters": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}}},
    {"type": "function", "function": {"name": "search_policy", "description": "Policy text for: returns, refunds or shipping.",
     "parameters": {"type": "object", "properties": {"topic": {"type": "string", "enum": ["returns", "refunds", "shipping"]}}, "required": ["topic"]}}},
    {"type": "function", "function": {"name": "escalate", "description": "Hand the conversation to a human agent.",
     "parameters": {"type": "object", "properties": {"reason": {"type": "string"}}, "required": ["reason"]}}},
]
print(f"🔧 {len(TOOLS)} tools: {', '.join(TOOLS)}")

In [ ]:
# ============================================================================
# THE AGENT LOOP: router span -> tool span(s) -> final answer
# ============================================================================
SYSTEM_PROMPT_V1 = "You are a customer-support agent. Use the tools when they help, then answer briefly."


def route(messages: list, system_prompt: str):
    """One routing decision, recorded as a CHAIN span named 'router'."""
    with tracer.start_as_current_span("router", openinference_span_kind="chain") as span:
        span.set_input(messages[-1]["content"] if messages[-1]["role"] == "user" else json.dumps(messages[-1], default=str))
        response = openai_client.chat.completions.create(
            model=MODEL, temperature=0,
            messages=[{"role": "system", "content": system_prompt}, *messages],
            tools=TOOL_SCHEMAS,
        )
        message = response.choices[0].message
        decision = message.tool_calls[0].function.name if message.tool_calls else "final_answer"
        span.set_output(decision)
        return message


@tracer.agent
def run_agent(question: str, system_prompt: str = SYSTEM_PROMPT_V1, max_steps: int = 4) -> dict:
    messages = [{"role": "user", "content": question}]
    path = []
    for _ in range(max_steps):
        message = route(messages, system_prompt)
        if not message.tool_calls:
            return {"answer": message.content, "path": path}
        messages.append(message)
        for call in message.tool_calls:
            result = TOOLS[call.function.name](**json.loads(call.function.arguments))
            path.append(call.function.name)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})
    return {"answer": "I could not finish this request.", "path": path}


print("✅ Agent ready")

### 1.2 🧪 Run it on six requests

Each request carries the **gold path** — the shortest correct tool sequence — used in Parts 3 and 4. Writing the gold path down before running is the same discipline as Module 3's gold trajectory.

In [ ]:
# ============================================================================
# TEST REQUESTS WITH GOLD PATHS
# ============================================================================
CASES = [
    {"question": "Where is my order 1042?",                          "gold_path": ["lookup_order"]},
    {"question": "Can I still return order 2077?",                   "gold_path": ["lookup_order", "search_policy"]},
    {"question": "How long do refunds take?",                        "gold_path": ["search_policy"]},
    {"question": "Do you ship to Canada?",                           "gold_path": ["search_policy"]},
    {"question": "Order 9999 never arrived and I want a person now.", "gold_path": ["lookup_order", "escalate"]},
    {"question": "I'd like to talk to a human, please.",             "gold_path": ["escalate"]},
]

for case in CASES:
    case["result"] = run_agent(case["question"])
    print(f"🤖 {case['question']:<52} path={case['result']['path']}")

### 1.3 🔍 Read the spans back

Open http://localhost:6006 and the `c01-13-support-agent` project to see each trace as a tree. The same spans come back as a DataFrame — one row per span — which is what every evaluation below runs on.

> **Note**: column names are Phoenix's flattened OpenInference attributes, e.g. `span_kind`, `name`, `attributes.input.value`, `attributes.output.value`, `context.span_id`, `context.trace_id`.

In [ ]:
# ============================================================================
# SPANS AS A DATAFRAME
# ============================================================================
spans = phoenix.spans.get_spans_dataframe(project_identifier=PROJECT)
if "context.span_id" not in spans.columns:   # the span id may arrive as the index
    spans = spans.reset_index()
print(f"📋 {len(spans)} spans in {spans['context.trace_id'].nunique()} traces")
print(Counter(spans["span_kind"]))
spans[["name", "span_kind", "attributes.input.value", "attributes.output.value"]].head(10)

---
## ⚖️ Part 2: Router and skill evaluations

Two different scorers for two different questions, the same split as Module 0's code-vs-judge decision:

- **Router** — *did it pick the right tool for this request?* Needs judgement about intent, so an LLM judge.
- **Skill** — *did the `lookup_order` tool return a well-formed answer?* Checkable by code, so code — free, deterministic.

Both results are logged **onto the spans they judge**, so the Phoenix UI shows the verdict next to the trace.

### 2.1 🧑‍⚖️ LLM judge on the router's choice

In [ ]:
# ============================================================================
# ROUTER EVAL: an LLM classifier over every 'router' span
# ============================================================================
from phoenix.evals import LLM, create_classifier, evaluate_dataframe
from phoenix.evals.utils import to_annotation_dataframe
from openinference.instrumentation import suppress_tracing

ROUTER_TEMPLATE = """You are checking a support agent's routing decision.
Tools: lookup_order (order status by ID), search_policy (returns/refunds/shipping policy),
escalate (hand to a human), final_answer (no tool needed now).

Customer message or latest step: {input}
Tool the agent chose: {output}

Was this a reasonable next step? Answer "correct" or "incorrect"."""

router_judge = create_classifier(
    name="router_correct",
    prompt_template=ROUTER_TEMPLATE,
    llm=LLM(provider="openai", model=MODEL),
    choices={"correct": 1.0, "incorrect": 0.0},
)

router_spans = spans[spans["name"] == "router"].rename(
    columns={"attributes.input.value": "input", "attributes.output.value": "output"}
)[["context.span_id", "input", "output"]]

# auto_instrument traces every OpenAI call, the judge's included. suppress_tracing keeps the
# judge's own LLM calls out of the agent's project, so evaluation never pollutes what it measures.
with suppress_tracing():
    router_results = evaluate_dataframe(dataframe=router_spans, evaluators=[router_judge])
router_annotations = to_annotation_dataframe(router_results)   # adds annotation name and kind columns
phoenix.spans.log_span_annotations_dataframe(dataframe=router_annotations)

print(f"🧑‍⚖️ Router judged on {len(router_spans)} decisions; annotations logged to Phoenix")
router_annotations[["label", "score", "explanation"]]

### 2.2 🧮 Code check on a skill's output

A tool span's output is data, so the check is a function. `lookup_order` must return JSON that either has a `status` or an explicit `error` — never free text, never an empty string. A judge here would be cost without information.

In [ ]:
# ============================================================================
# SKILL EVAL: a deterministic check on every lookup_order span
# ============================================================================
def lookup_output_is_valid(output: str) -> bool:
    try:
        data = json.loads(output)
    except (TypeError, json.JSONDecodeError):
        return False
    return "status" in data or "error" in data


lookup_spans = spans[spans["name"] == "lookup_order"]
skill_annotations = pd.DataFrame({
    "span_id": lookup_spans["context.span_id"].values,
    "label": ["valid" if lookup_output_is_valid(o) else "invalid" for o in lookup_spans["attributes.output.value"]],
})
skill_annotations["score"] = (skill_annotations["label"] == "valid").astype(float)
phoenix.spans.log_span_annotations_dataframe(
    dataframe=skill_annotations, annotation_name="lookup_output_valid", annotator_kind="CODE"
)
print(f"🧮 {int(skill_annotations['score'].sum())}/{len(skill_annotations)} lookup_order outputs valid")

---
## 🛤️ Part 3: Trajectory evaluation — the convergence score

Module 3 compared a trace against a gold trajectory step by step. The **convergence score** asks a coarser question that works on any stored trace: how close did the agent come to the shortest path?

$$\text{convergence} = \frac{\text{optimal path length}}{\text{actual path length}}$$

1.0 means it took the shortest route; 0.5 means twice as many steps as needed. It says nothing about *which* tools — a wrong single step scores 1.0 — so pair it with a path check, as Part 4 does.

**No gold path? Use the shortest one you have seen.** Run the same request several times, or several paraphrases of it, and treat the shortest path that reached a correct answer as optimal. It is an estimate — it can only be as good as the best run — but it lets convergence work on traffic nobody labelled. The paths come from the stored `TOOL` spans, ordered by start time, not from the agent's own return value: evaluate what was recorded, not what the code claims.

In [ ]:
# ============================================================================
# CONVERGENCE FROM STORED SPANS
# ============================================================================
tool_spans = spans[spans["span_kind"] == "TOOL"].sort_values("start_time")
paths_by_trace = tool_spans.groupby("context.trace_id")["name"].apply(list)

roots = spans[spans["span_kind"] == "AGENT"].set_index("context.trace_id")["attributes.input.value"]
gold_by_question = {c["question"]: c["gold_path"] for c in CASES}


def convergence(gold: list, actual: list) -> float:
    if not actual:
        return 0.0 if gold else 1.0
    return min(1.0, len(gold) / len(actual))


rows = []
for trace_id, question_json in roots.items():
    question = next((q for q in gold_by_question if q in str(question_json)), None)
    if question is None:
        continue
    actual = paths_by_trace.get(trace_id, [])
    gold = gold_by_question[question]
    rows.append({"question": question, "gold": gold, "actual": actual,
                 "path_match": actual == gold, "convergence": convergence(gold, actual)})

trajectory = pd.DataFrame(rows)
print(f"🧮 mean convergence {trajectory['convergence'].mean():.2f}; exact path match {trajectory['path_match'].mean():.0%}")
trajectory

For the deterministic, step-by-step version of this — loops, redundant calls, recovery after a failed call — see Module 3, [`12_Agent_Trajectory_Evaluation.ipynb`](12_Agent_Trajectory_Evaluation.ipynb).

---
## 🧪 Part 4: Experiments — compare two prompts on a fixed dataset

Parts 2–3 scored whatever traffic happened to arrive. An **experiment** fixes the inputs instead: a versioned dataset, a task (the agent), and evaluators, so two versions of the agent are compared on exactly the same cases. That makes a prompt change a measured decision rather than an impression — the same idea as MLflow's evaluation runs in course 02 ([`05_prompt_versioning_and_regression.ipynb`](../02-databricks-mlflow-evals/05_prompt_versioning_and_regression.ipynb)).

### 4.1 📦 A versioned dataset

In [ ]:
# ============================================================================
# DATASET: inputs + expected outputs, stored and versioned in Phoenix
# ============================================================================
dataset = phoenix.datasets.create_dataset(
    name=f"c01-13-support-routing-{datetime.now():%Y%m%d-%H%M%S}",   # dataset names are unique; re-runs get a new one
    inputs=[{"question": c["question"]} for c in CASES],
    outputs=[{"gold_path": c["gold_path"]} for c in CASES],
)
print(f"📦 Dataset with {len(CASES)} examples created")

### 4.2 🧪 Task, evaluators, two runs

The task runs the agent on one example. The evaluators receive `output` (what the task returned) and `expected` (the example's stored output) by argument name. The second run changes one thing — the system prompt tells the agent to check the order before answering about it, and to escalate when the customer asks for a person.

In [ ]:
# ============================================================================
# TASKS AND EVALUATORS
# ============================================================================
SYSTEM_PROMPT_V2 = (
    "You are a customer-support agent. If the customer names an order, look it up before "
    "answering. If they ask for a person or are angry about a lost order, escalate. "
    "Use search_policy for returns, refunds and shipping questions. Answer briefly."
)


def task_v1(input):
    return run_agent(input["question"], system_prompt=SYSTEM_PROMPT_V1)


def task_v2(input):
    return run_agent(input["question"], system_prompt=SYSTEM_PROMPT_V2)


def path_match(output, expected) -> bool:
    return output["path"] == expected["gold_path"]


def path_convergence(output, expected) -> float:
    return convergence(expected["gold_path"], output["path"])


EVALUATORS = [path_match, path_convergence]
runs = {
    name: phoenix.experiments.run_experiment(
        dataset=dataset, task=task, evaluators=EVALUATORS, experiment_name=name, print_summary=False
    )
    for name, task in [("prompt-v1", task_v1), ("prompt-v2", task_v2)]
}
print("✅ Two experiments recorded")

In [ ]:
# ============================================================================
# COMPARE THE TWO RUNS
# ============================================================================
def mean_scores(ran) -> dict:
    by_name = {}
    for run in ran["evaluation_runs"]:
        score = (run.result or {}).get("score")
        if score is not None:
            by_name.setdefault(run.name, []).append(score)
    return {name: sum(v) / len(v) for name, v in by_name.items()}


comparison = pd.DataFrame({name: mean_scores(ran) for name, ran in runs.items()})
print("🧮 Mean score per evaluator (open the dataset in Phoenix to compare runs example by example)")
comparison

**Reading the result:** with six examples a single flipped case moves `path_match` by 17 points, so treat the comparison as a smoke test, not a verdict — the sample-size argument is course 03's notebook 10. What the experiment *does* give you is the per-example diff in the Phoenix UI: which questions changed path between v1 and v2, and in which direction.

---
## 📝 Summary

In this notebook, we learned:

### 1. Tracing
- **Span kinds carry meaning**: `AGENT` per request, `CHAIN` per routing decision, `LLM` from auto-instrumentation, `TOOL` per call — evaluation later selects spans by kind and name.
- **Record decisions explicitly**: the router span's input and output make the decision judgeable without parsing raw LLM messages.

### 2. Router and skill evaluations
- **Judge intent, check data**: an LLM classifier for the routing choice; a plain function for the tool output's shape.
- **Write verdicts back onto spans** with `log_span_annotations_dataframe`, so the trace and its evaluation sit together.

### 3. Trajectory
- **Convergence = optimal ÷ actual path length**, computed from stored `TOOL` spans. It measures efficiency, not correctness — pair it with a path check.

### 4. Experiments
- **Dataset + task + evaluators** compare two agent versions on identical inputs; the evaluators bind `output` and `expected` by argument name.

### Next Steps
- Module 5: [`14_Capstone_CrewAI_Travel_Planner_Eval.ipynb`](14_Capstone_CrewAI_Travel_Planner_Eval.ipynb) — evaluating a real running multi-agent app.
- Online evaluation over production traffic, with no answer key: course 03, [`12_online_evaluation_and_the_loop.ipynb`](../03-agentic-eval-harness/12_online_evaluation_and_the_loop.ipynb).